In [1]:
from pathlib import Path
import subprocess
import torch

assert torch.cuda.is_available(), "Activa la GPU en Kaggle."

repo = Path("/kaggle/working/pa3-student")
datos = Path("/kaggle/input/datasets/bxigflor/pa3-dados/pa3_dados")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", "main",
            "https://github.com/nicolasspaniol/deep-learning-pa3.git",
            str(repo),
        ],
        check=True,
    )

for nombre in ("classes.csv", "labels", "images"):
    origen = datos / nombre
    destino = repo / "data" / nombre
    assert origen.exists(), f"Falta {origen}"
    if not destino.exists():
        destino.symlink_to(origen)

print("GPU:", torch.cuda.get_device_name(0))

Cloning into '/kaggle/working/pa3-student'...


GPU: Tesla T4


In [ ]:
%%bash
set -e
cd /kaggle/working/pa3-student
export PYTHONPATH="$PWD/src"

for threshold in 0.49766308069229126 0.7038832902908325 0.9445663690567017; do
  python -u -m deep_learning_pa3.train_student \
    --threshold "$threshold" --seed 42 --epochs 20
done

Device: cuda. Real labels: 105. Pseudo-labels: 4800.
  american_black_bear: 532
  american_crow: 392
  bird: 2
  bobcat: 25
  chicken: 56
  coyote: 43
  dog: 8
  domestic_cat: 445
  eastern_chipmunk: 67
  eastern_cottontail: 316
  eastern_fox_squirrel: 365
  eastern_gray_squirrel: 189
  grey_fox: 15
  horse: 2
  northern_raccoon: 196
  red_fox: 647
  striped_skunk: 373
  virginia_opossum: 482
  white_tailed_deer: 220
  wild_turkey: 244
  woodchuck: 181
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
Epoch 1: validation macro F1=0.1254
Epoch 2: validation macro F1=0.1969
Epoch 3: validation macro F1=0.2295
Epoch 4: validation macro F1=0.2142
Epoch 5: validation macro F1=0.2163
Epoch 6: validation macro F1=0.2238
Epoch 7: validation macro F1=0.1844
Epoch 8: validation macro F1=0.1716
Epoch 9: validation macro F1=0.2112
Epoch 10: validation macro F1=0.1668
Epoch 11: validation macro F1=0.1963
Epoch 12: va

Part 2


In [1]:
from pathlib import Path
import subprocess
import torch

assert torch.cuda.is_available(), "Activa una GPU en Kaggle."

repo = Path("/kaggle/working/pa3-parte2")
if not repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", "main",
            "https://github.com/nicolasspaniol/deep-learning-pa3.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)

assert (repo / "scripts/budget_curve.py").is_file(), "Sube el script a main."
print(torch.cuda.get_device_name(0))

Cloning into '/kaggle/working/pa3-parte2'...


Tesla T4


In [2]:
%%bash
set -e
cd /kaggle/working/pa3-parte2
export PYTHONPATH="$PWD/src"

python -u scripts/budget_curve.py \
  --data-dir /kaggle/input/datasets/bxigflor/pa3-dados/pa3_dados \
  --threshold 0.9445663690567017 \
  --epochs 20 \
  --output-dir /kaggle/working/parte2_results \
  --resume

Device: cuda
('baseline', 1, 42): 21 real, 0 pseudo-labels
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
  Epoch 1/20: val macro F1=0.0271
  Epoch 2/20: val macro F1=0.0469
  Epoch 3/20: val macro F1=0.0345
  Epoch 4/20: val macro F1=0.0379
  Epoch 5/20: val macro F1=0.0374
  Epoch 6/20: val macro F1=0.0457
  Epoch 7/20: val macro F1=0.0488
  Epoch 8/20: val macro F1=0.0764
  Epoch 9/20: val macro F1=0.0811
  Epoch 10/20: val macro F1=0.0825
  Epoch 11/20: val macro F1=0.0976
  Epoch 12/20: val macro F1=0.1190
  Epoch 13/20: val macro F1=0.1292
  Epoch 14/20: val macro F1=0.1466
  Epoch 15/20: val macro F1=0.1530
  Epoch 16/20: val macro F1=0.1569
  Epoch 17/20: val macro F1=0.1645
  Epoch 18/20: val macro F1=0.1776
  Epoch 19/20: val macro F1=0.1753
  Epoch 20/20: val macro F1=0.1686
  Best epoch: 18. Val macro F1: 0.1776
('student', 1, 42): 21 real, 1617 pseudo-labels
  Epoch 1/20: val macro F1=0.0

100%|██████████| 44.7M/44.7M [00:00<00:00, 230MB/s]


In [3]:
from pathlib import Path
from shutil import make_archive
from IPython.display import FileLink, display

resultados = Path(
    "/kaggle/working/parte2_results/threshold_0.9445663690567017_epochs_20"
)
pesos = resultados / "weights"

assert pesos.is_dir(), f"No se encontró {pesos}. Revisa el umbral y las épocas."
print(f"Checkpoints encontrados: {len(list(pesos.glob('*.pt')))} de 10")

zip_path = make_archive(
    "/kaggle/working/pa3_parte2_weights",
    "zip",
    root_dir=resultados,
    base_dir="weights",
)

%cd /kaggle/working
display(FileLink(Path(zip_path).name))

Checkpoints encontrados: 10 de 10
/kaggle/working


/kaggle/working/pa3_parte2_weights.zip

In [4]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

working = Path("/kaggle/working")
archivos = [
    "curva_orcamento.csv",
    "curva_orcamento.png",
    "ganho_por_classe_5.csv",
]

# Busca la carpeta de resultados sin depender del nombre del umbral.
carpetas = sorted({p.parent for p in working.rglob("curva_orcamento.csv")})
if len(carpetas) != 1:
    raise RuntimeError(
        f"Se esperaba una sola carpeta de resultados. Encontradas: {carpetas}. "
        "Si hay varias, usa la ruta indicada en la línea Results: de tu corrida."
    )

resultados = carpetas[0]
faltantes = [nombre for nombre in archivos if not (resultados / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan estos resultados en {resultados}: {faltantes}")

zip_path = working / "pa3_parte2_resultados.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as zip_file:
    for nombre in archivos:
        zip_file.write(resultados / nombre, arcname=nombre)

%cd /kaggle/working
display(FileLink(zip_path.name))

/kaggle/working


/kaggle/working/pa3_parte2_resultados.zip